In [ ]:
%load_ext autoreload
%autoreload 2

In [1]:
from hurobo.utils.notebook import setup_notebook

ROOT = setup_notebook()  # chdir to the repo root and set MUJOCO_GL=egl; must precede any mujoco import

# Teleoperating the Panda with a human hand clip (and the keyboard)

A tracked human hand drives the LIBERO Panda's end effector; the gripper follows the pinch. Object dynamics are not the point yet: a small yellow cube with a collider appears between the fingers at the moment the gripper closes, so there is something to carry.

- `hurobo.teleop`: `TableToWorld` (hand deltas in the table frame -> world, anchored on the first pose), `yaw_aligning_views`, command sources (`HandCommandSource`, `KeyboardCommandSource`), `TeleopSession`.
- `hurobo.props`: `CubeProp`; `hurobo.teleop_app`: stand-in clip preparation, frame composer, `run_hand_teleop`.
- `hurobo.utils.video`: `slow_down_video` / `FrameReel` (optical-flow slow motion; the 1x clip is kept next to the slowed one).

**Stand-in data.** The hand is a real hand from a third-party GIF (`data/raw/overview_gif/hand_moves_cube/`, 18 frames, 1.7 s; not committed). The ArUco markers are *synthetic*, pasted in at a chosen camera pose with guessed intrinsics (fx = fy = 800 px). The marker, not a ChArUco board, gives the camera pose for every clip; ChArUco is only needed once per phone for intrinsics. Because the markers are fake, the extractor's QC gates are expected to flag the clip and depth is not anchored to the "bowl".

Live window and keyboard driving need a display, so they are scripts: `scripts/13_teleop.py hand <frames> --live` and `scripts/13_teleop.py keyboard`.

## The human clip in the table frame

In [2]:
import tempfile
from pathlib import Path

from hurobo.teleop_app import prepare_stand_in_clip
from hurobo.utils.notebook import show_video
from hurobo.utils.viewer import trajectories_figure, trajectory_figure

WORK_DIR = Path(tempfile.mkdtemp(prefix="hurobo_teleop_"))
CLIP_DIR = ROOT / "data/raw/overview_gif/hand_moves_cube"

clip = prepare_stand_in_clip(CLIP_DIR, WORK_DIR / "front", camera_position_table=(0.0, -0.45, 0.40))
demo = clip.demo
print(f"{len(demo.timestamps_s)} samples over {demo.timestamps_s[-1]:.2f} s")
print("QC:", "; ".join(clip.report.reasons))
trajectory_figure(
    demo.tcp_pos_table, demo.is_grasping, "Hand TCP in the table frame (stand-in markers)"
).show()

INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
W0000 00:00:1791723902.051043 1452213 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1791723902.060606 1452213 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1791723902.111352 1452212 landmark_projection_calculator.cc:81] Using NORM_RECT without IMAGE_DIMENSIONS is only supported for the square ROI. Provide IMAGE_DIMENSIONS or use PROJECTION_MATRIX.


31 samples over 1.50 s
QC: no release detected; grasp 0.15 m from bowl marker


## Different camera poses

The phone and the sim camera never share a viewpoint. Everything is expressed in the *table* frame (from the marker), so the hand path does not depend on where the camera was, only on how the camera sees it. Same pixels, but with the stand-in camera moved 90 degrees around the table, the table-frame path rotates accordingly; `yaw_aligning_views` then rotates the table into the sim world so that the motion looks the same from the sim's agentview camera as it did from the phone.

In [3]:
clip_side = prepare_stand_in_clip(CLIP_DIR, WORK_DIR / "side", camera_position_table=(0.45, 0.0, 0.40))
trajectories_figure(
    {
        "camera in front (0, -0.45, 0.4)": demo.tcp_pos_table,
        "camera at the side (0.45, 0, 0.4)": clip_side.demo.tcp_pos_table,
    },
    "Same pixels, two assumed camera poses (table frame)",
).show()

W0000 00:00:1791723903.192859 1452340 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1791723903.204741 1452342 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


## Teleoperate in the sim

The robot starts wherever LIBERO puts it; the first hand sample is pinned to that pose and everything after is a *delta*. The human motion is slowed 4x for the robot (it tracks better) and the video; the 1x video keeps every 4th frame so it lasts as long as the original clip. The display is a grid: the human clip (red cross = hand TCP, cyan ring = the robot's gripper re-projected through the human camera; they should sit on top of each other) and the *twin view* (a sim camera at the human camera's pose) on top; the agentview, the stock gallery view of the whole setup and the top-down wrist camera below.

In [4]:
import numpy as np

from hurobo.props import CubeProp
from hurobo.sim import LiberoTask
from hurobo.teleop_app import run_hand_teleop, write_run_videos, write_standin_slow_video

SLOWDOWN = 4.0
task = LiberoTask("libero_goal", 8)
result = run_hand_teleop(task, CubeProp(), clip, slowdown=SLOWDOWN, panel_height=240)
trace = result.trace
error_cm = 100 * trace.tracking_error_m()
print(
    f"{len(trace.t_s)} steps ({trace.t_s[-1]:.1f} s); table->world yaw {np.degrees(result.yaw_rad):.0f} deg"
)
print(f"tracking error mean {error_cm.mean():.1f} cm, max {error_cm.max():.1f} cm")
print(f"cube spawned at step {trace.cube_spawn_step}")
videos = write_run_videos(trace, WORK_DIR, SLOWDOWN)
videos["human_slow"] = write_standin_slow_video(clip, WORK_DIR / "front", SLOWDOWN)

Creating LIBERO envs | suites=['libero_goal'] | n_envs(per task)=1 | init_states=True
Restricting to task_ids=[8]
[info] using task orders [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
Built vec env | suite=libero_goal | task_id=8 | n_envs=1
Local assets not found. Downloading from HuggingFace Hub...


  self.qpos[addr] = value


131 steps (6.6 s); table->world yaw 90 deg
tracking error mean 1.2 cm, max 3.6 cm
cube spawned at step 38


In [5]:
from hurobo.utils.viewer import teleop_tracking_figure

teleop_tracking_figure(trace, "Hand-driven teleoperation: target vs end effector").show()

Real speed (1x), then slowed down 4x (the slowed run was executed slowly, so each frame is a real 20 Hz control step and the motion is smooth):

In [6]:
show_video(videos["1x"], width=900)

In [7]:
show_video(videos["slow"], width=900)

The slowed human clip on its own (optical-flow in-between frames; the original `standin_1x.mp4` is untouched):

In [8]:
show_video(videos["human_slow"], width=480)

## Keyboard teleoperation (scripted keys)

The same session, but the command source is the keyboard. Here the key presses come from a schedule so the notebook runs headless: hold `right` (+y in the agentview), then `down` (+x), press `space` to close the gripper (the cube appears), hold `w` to lift, `escape` to end. Interactively: `scripts/13_teleop.py keyboard`.

In [9]:
import numpy as np

from hurobo.teleop import KeyboardCommandSource, ScriptedKeys, TeleopSession, WorkspaceLimits
from hurobo.teleop_app import TeleopRunner, reset_with_props

cube = CubeProp()
obs = reset_with_props(task, cube)
p_world_eef0, _ = task.get_eef_pose(obs)
keys = ScriptedKeys(
    holds=[(5, 25, "right"), (30, 40, "down"), (60, 80, "w")], presses=[(45, "space"), (95, "escape")]
)
session = TeleopSession(
    task, KeyboardCommandSource(keys, p_world_eef0), WorkspaceLimits.around(p_world_eef0), cube
)
keyboard_trace = TeleopRunner(session).run(obs)
path = np.asarray(keyboard_trace.p_world_eef)
print(f"{len(path)} steps; end effector moved {np.round(path[-1] - p_world_eef0, 3)} m")
print(f"cube spawned at step {keyboard_trace.cube_spawn_step}")
trajectories_figure(
    {"end effector": path, "cube": np.asarray(keyboard_trace.p_world_cube)},
    "Keyboard-driven path (world frame)",
).show()

95 steps; end effector moved [0.084 0.158 0.159] m
cube spawned at step 45


## Camera sources and the live pipeline

`scripts/13_teleop.py hand SOURCE` accepts an image folder or video (replayed offline, as above) or a *live* source: `webcam:N` / `/dev/videoN` (also a phone bridged by scrcpy, DroidCam or USB webcam mode), a stream URL (`http://…`, `rtsp://…`) or `file:<video>`. A live source is processed frame by frame: table marker averaged and locked, hand tracked and smoothed, gripper from a streaming pinch hysteresis, first hand pose anchored to the robot pose (`r` re-anchors). Phone setup recipes: `docs/camera_sources.md`.

Below, the live pipeline is fed by the stand-in clip played as if it were a camera (`FileFrameSource`), which also shows how a source string is interpreted and what this machine's V4L2 devices are.

In [10]:
from hurobo.camera_source import FileFrameSource, list_cameras, parse_source

for text in ["webcam:0", "/dev/video10", "http://192.168.1.20:8080/video", f"file:{clip.video_path.name}"]:
    print(f"{text:36s} -> {parse_source(text)}")
for camera in list_cameras():
    print(camera)

webcam:0                             -> SourceSpec(kind='webcam', target=0)
/dev/video10                         -> SourceSpec(kind='webcam', target='/dev/video10')
http://192.168.1.20:8080/video       -> SourceSpec(kind='stream', target='http://192.168.1.20:8080/video')
file:standin_1x.mp4                  -> SourceSpec(kind='file', target='standin_1x.mp4')
CameraInfo(device='/dev/video0', name='HP 5MP Camera: HP 5MP Camera', is_capture_node=True)
CameraInfo(device='/dev/video1', name='HP 5MP Camera: HP 5MP Camera', is_capture_node=False)
CameraInfo(device='/dev/video2', name='HP 5MP Camera: HP IR Camera', is_capture_node=True)
CameraInfo(device='/dev/video3', name='HP 5MP Camera: HP IR Camera', is_capture_node=False)


In [11]:
from hurobo.teleop_app import run_live_teleop

live = run_live_teleop(
    task,
    CubeProp(),
    FileFrameSource(clip.video_path),
    clip.intrinsics,
    clip.layout,
    calibration_frames=3,
    panel_height=240,
)
live_trace = live.trace
print(f"{len(live_trace.t_s)} steps; final pipeline status: {live.source.status!r}")
print(f"table-to-world yaw {np.degrees(live.source.mapping.yaw_rad):.0f} deg")
teleop_tracking_figure(live_trace, "Live pipeline (clip as camera): target vs end effector").show()

W0000 00:00:1791723916.988213 1453048 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1791723916.995891 1453050 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


18 steps; final pipeline status: 'hand lost: holding'
table-to-world yaw 90 deg


### Without markers (relative pose only)

`layout=None` (script: `--no-markers`) drops the table marker. The camera frame becomes the reference: the first hand pose anchors the robot and only deltas count. The camera is taken to be level and facing the way the agentview does, so image right/up/away map to the agentview's right/up/away, gravity-aligned. This is the mode for a quick webcam test.

In [12]:
markerless = run_live_teleop(
    task, None, FileFrameSource(clip.video_path), clip.intrinsics, None, panel_height=240
)
mapping = markerless.source.mapping
target = np.asarray(markerless.trace.p_world_target)
hand_delta_cam = mapping.point_to_table(target[-1]) - mapping.point_to_table(target[0])
print("hand delta in camera axes (right, down, away) cm:", np.round(100 * hand_delta_cam, 1))
print("robot target delta in the world (x, y, z) cm:    ", np.round(100 * (target[-1] - target[0]), 1))

W0000 00:00:1791723918.406453 1453169 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1791723918.415039 1453169 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


hand delta in camera axes (right, down, away) cm: [ 5.7 -7.9  9. ]
robot target delta in the world (x, y, z) cm:     [-9.   5.7  7.9]


In [13]:
task.close()